# Import Library

In [676]:
from pynq import Overlay, allocate
import threading
import time
import mmap, struct, time
import json, os
import numpy as np

# Import Files

In [677]:
MODEL_PATH = os.path.abspath("model")
DATA_PATH = os.path.abspath("data")
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

WEIGHT_MEM_FILE = os.path.join(MODEL_PATH, "G_d5_Q9.14_decoder_weight.mem")
BIAS_MEM_FILE   = os.path.join(MODEL_PATH, "G_d5_Q9.14_decoder_bias.mem")
IFMAP_L0_FILE   = os.path.join(DATA_PATH, "b3_1.mem")
IFMAP_L1_FILE   = os.path.join(DATA_PATH, "d2_in.mem")
IFMAP_L2_FILE   = os.path.join(DATA_PATH, "d3_in.mem")
IFMAP_L3_FILE   = os.path.join(DATA_PATH, "d4_in.mem")

L0_RESULT   = os.path.join(DATA_PATH, "d1.mem")
L1_RESULT   = os.path.join(DATA_PATH, "d2_out.mem")
L2_RESULT   = os.path.join(DATA_PATH, "d3_out.mem")
L3_RESULT   = os.path.join(DATA_PATH, "d4_out.mem")

# ----- Offset dalam file .mem gabungan -----
WEIGHT_OFFSET_L0 = 0
WEIGHT_OFFSET_L1 = 131072
WEIGHT_OFFSET_L2 = 196608
WEIGHT_OFFSET_L3 = 212992

BIAS_OFFSET_L0 = 0
BIAS_OFFSET_L1 = 8192
BIAS_OFFSET_L2 = 16384
BIAS_OFFSET_L3 = 24576

# Ensuring Correct Data

In [678]:
def load_mem_file(filepath):
    """Load Verilog $readmemh .mem file → numpy uint32 array."""
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.split('//')[0].strip()
            if not line or line.startswith('@'):
                continue
            values.append(int(line, 16))
    return np.array(values, dtype=np.uint32)

In [679]:
print("Loading weight...", end=' ', flush=True)
weight_data = load_mem_file(WEIGHT_MEM_FILE)
print(f"{len(weight_data)} words")

print("Loading bias...", end=' ', flush=True)
bias_data = load_mem_file(BIAS_MEM_FILE)
print(f"{len(bias_data)} words")

print("Loading ifmap L0...", end=' ', flush=True)
ifmap_l0 = load_mem_file(IFMAP_L0_FILE)
print(f"{len(ifmap_l0)} words")

print("Loading ifmap L1...", end=' ', flush=True)
ifmap_l1 = load_mem_file(IFMAP_L1_FILE)
print(f"{len(ifmap_l1)} words")

print("Loading ifmap L2...", end=' ', flush=True)
ifmap_l2 = load_mem_file(IFMAP_L2_FILE)
print(f"{len(ifmap_l2)} words")

print("Loading ifmap L3...", end=' ', flush=True)
ifmap_l3 = load_mem_file(IFMAP_L3_FILE)
print(f"{len(ifmap_l3)} words")

print("Loading L0 Result...", end=' ', flush=True)
result_l0 = load_mem_file(L0_RESULT)
print(f"{len(result_l0)} words")

print("Loading L1 Result...", end=' ', flush=True)
result_l1 = load_mem_file(L1_RESULT)
print(f"{len(result_l1)} words")

print("Loading L2 Result...", end=' ', flush=True)
result_l2 = load_mem_file(L2_RESULT)
print(f"{len(result_l2)} words")

print("Loading L3 Result...", end=' ', flush=True)
result_l3 = load_mem_file(L3_RESULT)
print(f"{len(result_l3)} words")

Loading weight... 217088 words
Loading bias... 32768 words
Loading ifmap L0... 8192 words
Loading ifmap L1... 16384 words
Loading ifmap L2... 16384 words
Loading ifmap L3... 16384 words
Loading L0 Result... 8192 words
Loading L1 Result... 8192 words
Loading L2 Result... 8192 words
Loading L3 Result... 8192 words


In [680]:
def load_mem_file_2d(filepath, channels, time_len, dtype=np.uint32):
    values = []
    with open(filepath, 'r') as f:
        for line in f:
            line = line.split('//')[0].strip()
            if not line or line.startswith('@'):
                continue
            values.append(int(line, 16))

    arr = np.array(values, dtype=dtype)

    expected_size = channels * time_len
    if arr.size != expected_size:
        raise ValueError(
            f"Size mismatch in {filepath}: "
            f"expected {expected_size}, got {arr.size}"
        )

    # reshape to (time, channel)
    arr = arr.reshape((time_len, channels))

    # transpose to (channel, time)
    arr = arr.T

    return arr

In [681]:
ch_pos_per_layer = [
    (128, 64),
    (64, 128),
    (32, 256),
    (16, 512)
]

# ---- Load IFMAPS ----

print("Loading IFMAP L0 2D...", flush=True)
ifmap0_2d = load_mem_file_2d(IFMAP_L0_FILE, 256, 32)
print("Shape:", ifmap0_2d.shape)

print("Loading IFMAP L1 2D...", flush=True)
ifmap1_2d = load_mem_file_2d(IFMAP_L1_FILE, 256, 64)  # doubled channel
print("Shape:", ifmap1_2d.shape)

print("Loading IFMAP L2 2D...", flush=True)
ifmap2_2d = load_mem_file_2d(IFMAP_L2_FILE, 128, 128)   # doubled channel
print("Shape:", ifmap2_2d.shape)

print("Loading IFMAP L3 2D...", flush=True)
ifmap3_2d = load_mem_file_2d(IFMAP_L3_FILE, 64, 256)   # doubled channel
print("Shape:", ifmap3_2d.shape)


# ---- Load RESULTS ----

print("Loading RESULT L0 2D...", flush=True)
result0_2d = load_mem_file_2d(L0_RESULT, 128, 64)
print("Shape:", result0_2d.shape)

print("Loading RESULT L1 2D...", flush=True)
result1_2d = load_mem_file_2d(L1_RESULT, 64, 128)
print("Shape:", result1_2d.shape)

print("Loading RESULT L2 2D...", flush=True)
result2_2d = load_mem_file_2d(L2_RESULT, 32, 256)
print("Shape:", result2_2d.shape)

print("Loading RESULT L3 2D...", flush=True)
result3_2d = load_mem_file_2d(L3_RESULT, 16, 512)
print("Shape:", result3_2d.shape)

Loading IFMAP L0 2D...
Shape: (256, 32)
Loading IFMAP L1 2D...
Shape: (256, 64)
Loading IFMAP L2 2D...
Shape: (128, 128)
Loading IFMAP L3 2D...
Shape: (64, 256)
Loading RESULT L0 2D...
Shape: (128, 64)
Loading RESULT L1 2D...
Shape: (64, 128)
Loading RESULT L2 2D...
Shape: (32, 256)
Loading RESULT L3 2D...
Shape: (16, 512)


In [682]:
def compare_prefix_2d(ifmap_2d, result_2d, name=""):
    """
    Compare first half channels of ifmap with result.
    Assumes:
        ifmap shape = (2C, T)
        result shape = (C, T)
    """

    C_res, T_res = result_2d.shape
    C_if, T_if = ifmap_2d.shape

    if T_res != T_if:
        print(f"[{name}] [ERROR] Time dimension mismatch")
        return False

    if C_if < C_res:
        print(f"[{name}] [ERROR] Ifmap has fewer channels than result")
        return False

    prefix = ifmap_2d[:C_res, :]

    if np.array_equal(prefix, result_2d):
        print(f"[{name}] [OPTIMAL] Prefix matches perfectly")
        return True
    else:
        diff = np.where(prefix != result_2d)
        print(f"[{name}] [ERROR] Mismatch at {len(diff[0])} positions")
        print("First 10 mismatches:")
        for i in range(min(10, len(diff[0]))):
            c = diff[0][i]
            t = diff[1][i]
            print(f"  (ch={c}, t={t}) "
                  f"ifmap={prefix[c,t]:08X}, "
                  f"result={result_2d[c,t]:08X}")
        return False

In [683]:
def get_appended_channels_2d(ifmap_2d, result_2d):
    """
    Extract appended channels from ifmap.
    Returns channels beyond result channels.
    """

    C_res = result_2d.shape[0]
    appended = ifmap_2d[C_res:, :]
    return appended

In [684]:
# L1 vs L0
compare_prefix_2d(ifmap1_2d, result0_2d, "L1 vs L0")
append1 = get_appended_channels_2d(ifmap1_2d, result0_2d)
print("L1 appended shape:", append1.shape)


# L2 vs L1
compare_prefix_2d(ifmap2_2d, result1_2d, "L2 vs L1")
append2 = get_appended_channels_2d(ifmap2_2d, result1_2d)
print("L2 appended shape:", append2.shape)


# L3 vs L2
compare_prefix_2d(ifmap3_2d, result2_2d, "L3 vs L2")
append3 = get_appended_channels_2d(ifmap3_2d, result2_2d)
print("L3 appended shape:", append3.shape)

[L1 vs L0] [OPTIMAL] Prefix matches perfectly
L1 appended shape: (128, 64)
[L2 vs L1] [OPTIMAL] Prefix matches perfectly
L2 appended shape: (64, 128)
[L3 vs L2] [OPTIMAL] Prefix matches perfectly
L3 appended shape: (32, 256)


# Loading Hardware

In [685]:
decoder_imp_path = os.path.abspath("decoder_implementation")
os.makedirs(decoder_imp_path, exist_ok=True)

overlay = Overlay(os.path.join(decoder_imp_path, "design_1.bit"))
# overlay.ip_dict

In [686]:
print(overlay.ip_dict.keys())
print(overlay.gpio_dict.keys())

dict_keys(['axi_dma_0', 'axi_dma_1', 'axi_dma_2', 'processing_system7_0'])
dict_keys([])


In [687]:
dma_weight = overlay.axi_dma_0
dma_ifmap  = overlay.axi_dma_1
dma_bias   = overlay.axi_dma_2

# reset_gpio = overlay.axi_gpio_0
# reset_gpio.write(0, 0x1)  # aresetn = 0
# time.sleep(0.01)
# reset_gpio.write(1, 0x1)  # aresetn = 1
# time.sleep(0.01)

# Identify the "Listener" channels
dma_receipt = dma_weight.recvchannel 
dma_results = dma_ifmap.recvchannel

# Utility Data Load to FPGA

## Packet Builder

In [688]:
# ============================================================
# CELL 4: Helper — Packet Builder -> OUT NP ARRAY
# ============================================================

def _header(layer_id, num_words_per_bram):
    header = [0x00C0DE, 0x000001, layer_id, 15, 0, int(num_words_per_bram)]
    return [val & 0xFFFFFF for val in header]

# ---- WEIGHT packets ----

def build_weight_l0(batch_id, weight):
    """
    Layer 0 weight packet, 1 batch.
    Per batch: 16 BRAMs × 1024 words = 16384 words.
    OC base = batch_id × 16.
    DDR: OFFSET_L0 + (oc × 1024) + (k × 256) + ich
    """
    buf = _header(0, 1024)
    for bram_id in range(16):
        k_pos      = bram_id & 0x3
        oc_in_tile = (bram_id >> 2) & 0x3
        for tile in range(4):
            oc_abs = batch_id * 16 + tile * 4 + oc_in_tile
            for ich in range(256):
                addr = WEIGHT_OFFSET_L0 + oc_abs * 1024 + k_pos * 256 + ich
                buf.append(int(weight[addr]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)


# ---- IFMAP packets ----

def build_ifmap_l0(ifmap):
    """
    Layer 0 IFMAP: 32pos × 256ch = 8192 words.
    Stride 16: BRAM_n menyimpan pos [n, n+16] × 256ch.
    DDR layout: pos × 256 + ch
    """
    buf = _header(0, 512)   # 512 words per BRAM
    for bram_id in range(16):
        for pos_group in range(2):
            position = bram_id + pos_group * 16
            for ch in range(256):
                idx = position * 256 + ch
                buf.append(int(ifmap[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)


# ---- BIAS packets ----

def build_bias_l0(bias):
    """
    Layer 0 bias: 128ch × 64pos = 8192 words.
    BRAM_n: Ch[n, n+16, ..., n+112] (8 page) × 64pos.
    DDR: OFFSET_L0 + ch × 64 + pos
    """
    buf = _header(0, 512)
    for bram_id in range(16):
        for page in range(8):
            ch = page * 16 + bram_id
            for pos in range(64):
                idx = BIAS_OFFSET_L0 + ch * 64 + pos
                buf.append(int(bias[idx]) & 0xFFFFFF)
    return np.array(buf, dtype=np.uint32)

print("Packet builder functions defined")

Packet builder functions defined


In [689]:
# ============================================================
# CELL 7: Pre-build semua paket
# ============================================================
import time

t0 = time.time()
print("Building packets...")

# Layer 0 — 8 batches weight
print("  Layer 0 weights (8 batches)...", end=' ', flush=True)
pkt_w_l0 = [build_weight_l0(b, weight_data) for b in range(8)]
print(f"{sum(len(p) for p in pkt_w_l0):,} words total")

print("  Layer 0 ifmap...", end=' ', flush=True)
pkt_i_l0 = build_ifmap_l0(ifmap_l0)
print(f"{len(pkt_i_l0):,} words")

print("  Layer 0 bias...", end=' ', flush=True)
pkt_b_l0 = build_bias_l0(bias_data)
print(f"{len(pkt_b_l0):,} words")

print(f"\nBuild selesai dalam {time.time()-t0:.1f} detik")

Building packets...
  Layer 0 weights (8 batches)... 131,120 words total
  Layer 0 ifmap... 8,198 words
  Layer 0 bias... 8,198 words

Build selesai dalam 1.3 detik


## Sending Utility

In [690]:
def dma_send_async(dma, packet_np):
    '''
    == Example Usage == 
    buf_handle = dma_send_async(dma_weight, pkt_w_l0[0])
    dma_weight.sendchannel.wait() 
    buf_handle.freebuffer()
    '''
    buf = allocate(shape=(len(packet_np),), dtype=np.uint32)
    np.copyto(buf, packet_np)

    dma.sendchannel.transfer(buf)
    
    return buf

def to_signed_24bit(arr):
    """Convert 24-bit unsigned integers to signed 32-bit integers."""
    # Mask to ensure only 24 bits are considered
    a = np.array(arr, dtype=np.int32) & 0xFFFFFF
    # If the 24th bit is 1, subtract 2^24 to get the negative value
    return np.where(a >= 0x800000, a - 0x1000000, a)

# Execution Helper Functions

In [691]:
def hard_start_channel(dma, channel_name):
    """
    Paksa start channel via register langsung, bypass PYNQ state machine.
    channel_name: 'send' atau 'recv'
    """
    if channel_name == 'send':
        cr_offset = 0x00
        sr_offset = 0x04
    else:
        cr_offset = 0x30
        sr_offset = 0x34

    # 1. Reset channels
    dma.write(cr_offset, 0x4)
    timeout = 0
    while (dma.read(cr_offset) & 0x4) and timeout < 10000:
        timeout += 1
    
    # 2. Set RS=1 (Run/Stop bit) to register
    current_cr = dma.read(cr_offset)
    dma.write(cr_offset, current_cr | 0x1)
    
    sr = dma.read(sr_offset)
    running = (dma.read(cr_offset) & 0x1) == 1
    halted  = (sr & 0x1)
    print(f"  {channel_name}: CR=0x{dma.read(cr_offset):08X} SR=0x{sr:08X} running={running} halted={halted}")

def full_dma_reset(dma, name, has_recv=True):
    print(f"Resetting {name}...")
    hard_start_channel(dma, 'send')
    if has_recv:
        hard_start_channel(dma, 'recv')
    
    try:
        dma.sendchannel._flush_before_transfer = True
        dma.sendchannel._first_transfer = True
        if has_recv and dma.recvchannel is not None:
            dma.recvchannel._flush_before_transfer = True
            dma.recvchannel._first_transfer = True
    except Exception as e:
        print(f"  PYNQ state sync warning: {e}")

def pynq_proper_reset(dma, name, has_recv=True):
    """
    Reset DMA menggunakan API PYNQ native sepenuhnya.
    Tidak ada raw register write untuk CR.
    """
    print(f"Resetting {name}...")
    
    # Stop if running
    try:
        dma.sendchannel.stop()
        print(f"  send: stopped")
    except Exception as e:
        print(f"  send stop: {e}")
    
    if has_recv and dma.recvchannel is not None:
        try:
            dma.recvchannel.stop()
            print(f"  recv: stopped")
        except Exception as e:
            print(f"  recv stop: {e}")
    
    # Reset internal PYNQ state flags
    dma.sendchannel._first_transfer = True
    if has_recv and dma.recvchannel is not None:
        dma.recvchannel._first_transfer = True
    
    # Start via PYNQ API
    try:
        dma.sendchannel.start()
        print(f"  send: started, running={dma.sendchannel.running}")
    except Exception as e:
        print(f"  send start ERROR: {e}")
    
    if has_recv and dma.recvchannel is not None:
        try:
            dma.recvchannel.start()
            print(f"  recv: started, running={dma.recvchannel.running}")
        except Exception as e:
            print(f"  recv start ERROR: {e}")

def reset_dma(dma):
    # 1. Hardware Reset (bit 2 of Control Register)
    # Offset 0x00 for Send (MM2S), Offset 0x30 for Receive (S2MM)
    dma.write(0x00, 0x4)  
    dma.write(0x30, 0x4)
    
    # 2. Wait for reset bit to clear (hardware clears it when done)
    timeout = 0
    while (dma.read(0x00) & 0x4) and timeout < 1000:
        timeout += 1
    
    # 3. CRITICAL: Restart the DMA channels
    # PYNQ channels have a .start() method that sets the RS (Run/Stop) bit
    dma.sendchannel.start()
    dma.recvchannel.start()
    
    print(f"DMA {dma} reset and restarted.")

In [692]:
def dma_send(dma, packet_np):
    buf = allocate(shape=(len(packet_np),), dtype=np.uint32)
    np.copyto(buf, packet_np)
    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()
    buf.freebuffer()

def dma_send_async(dma, packet_np):
    buf = allocate(shape=(len(packet_np),), dtype=np.uint32)
    np.copyto(buf, packet_np)
    dma.sendchannel.transfer(buf)
    return buf

def pynq_proper_reset(dma, name, has_recv=True):
    try: dma.sendchannel.stop()
    except: pass
    if has_recv and dma.recvchannel is not None:
        try: dma.recvchannel.stop()
        except: pass
    dma.sendchannel._first_transfer = True
    if has_recv and dma.recvchannel is not None:
        dma.recvchannel._first_transfer = True
    dma.sendchannel.start()
    if has_recv and dma.recvchannel is not None:
        dma.recvchannel.start()
    print(f"{name}: send.running={dma.sendchannel.running}", end="")
    if has_recv and dma.recvchannel:
        print(f" recv.running={dma.recvchannel.running}", end="")
    print()

print("Helpers defined.")

Helpers defined.


In [693]:
def decode_output(m0_data, m1_data, num_ch, num_pos):
    """
    Susun ulang flat buffer m0/m1 menjadi array [ch, pos].
    m0_data, m1_data: numpy int32, panjang 4096 masing-masing.
    num_ch  : jumlah channel output layer ini
    num_pos : jumlah posisi output layer ini
    Setiap BRAM memiliki 512 entry.
    """
    output = np.zeros((num_ch, num_pos), dtype=np.int32)
    for ch in range(num_ch):
        bram_id  = ch % 16
        page     = ch // 16
        for pos in range(num_pos):
            bram_addr = page * num_pos + pos
            if bram_id < 8:
                output[ch, pos] = m0_data[bram_id * 512 + bram_addr]
            else:
                output[ch, pos] = m1_data[(bram_id - 8) * 512 + bram_addr]
    return output


def save_output_perchannel(output, filename, layer_name):
    """
    Simpan output [num_ch, num_pos] ke file teks per-channel,
    format sama dengan testbench (satu nilai per baris).
    """
    num_ch, num_pos = output.shape
    with open(filename, 'w') as f:
        f.write("=================================================\n")
        f.write(f"{layer_name} OUTPUT - PER CHANNEL DUMP\n")
        f.write(f"Total: {num_ch} Channels x {num_pos} Positions\n")
        f.write("=================================================\n")
        for ch in range(num_ch):
            f.write(f"\n=== CHANNEL {ch} ===\n")
            for pos in range(num_pos):
                f.write(f"{int(output[ch, pos])}\n")
        f.write("\n=================================================\n")
    print(f"  Saved: {filename}")


# Execution

In [694]:
# Setup
full_dma_reset(dma_weight, "dma_weight", has_recv=True)
full_dma_reset(dma_ifmap,  "dma_ifmap",  has_recv=True)
full_dma_reset(dma_bias,   "dma_bias",   has_recv=False)

print("\n=== VERIFIKASI AKHIR ===")
for name, dma in [("dma_weight", dma_weight), ("dma_ifmap", dma_ifmap), ("dma_bias", dma_bias)]:
    mm2s_cr = dma.read(0x00)
    s2mm_cr = dma.read(0x30)
    mm2s_sr = dma.read(0x04)
    s2mm_sr = dma.read(0x34)
    print(f"{name}: MM2S RS={mm2s_cr&1} SR=0x{mm2s_sr:08X} | S2MM RS={s2mm_cr&1} SR=0x{s2mm_sr:08X}")
    
    # Cek error flags
    if mm2s_sr & 0x70:
        print(f"  ⚠ MM2S ERROR FLAGS: IntErr={(mm2s_sr>>4)&1} SlvErr={(mm2s_sr>>5)&1} DecErr={(mm2s_sr>>6)&1}")
    if s2mm_sr & 0x70:
        print(f"  ⚠ S2MM ERROR FLAGS: IntErr={(s2mm_sr>>4)&1} SlvErr={(s2mm_sr>>5)&1} DecErr={(s2mm_sr>>6)&1}")

pynq_proper_reset(dma_weight, "dma_weight", has_recv=True)
pynq_proper_reset(dma_ifmap,  "dma_ifmap",  has_recv=True)
pynq_proper_reset(dma_bias,   "dma_bias",   has_recv=False)

print("\n=== VERIFIKASI ===")
for name, dma, has_recv in [
    ("dma_weight", dma_weight, True),
    ("dma_ifmap",  dma_ifmap,  True),
    ("dma_bias",   dma_bias,   False)
]:
    mm2s = dma.read(0x00) & 1
    s2mm = dma.read(0x30) & 1
    mm2s_sr = dma.read(0x04)
    s2mm_sr = dma.read(0x34)
    print(f"{name}: MM2S RS={mm2s} | S2MM RS={s2mm} | send.running={dma.sendchannel.running}")
    if has_recv and dma.recvchannel:
        print(f"         recv.running={dma.recvchannel.running}")
    if mm2s_sr & 0x70:
        print(f"  ⚠ MM2S ERR: 0x{mm2s_sr:08X}")
    if s2mm_sr & 0x70:
        print(f"  ⚠ S2MM ERR: 0x{s2mm_sr:08X}")

Resetting dma_weight...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0
  recv: CR=0x00010003 SR=0x00000000 running=True halted=0
Resetting dma_ifmap...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0
  recv: CR=0x00010003 SR=0x00000000 running=True halted=0
Resetting dma_bias...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0

=== VERIFIKASI AKHIR ===
dma_weight: MM2S RS=0 SR=0x00000001 | S2MM RS=1 SR=0x00000000
dma_ifmap: MM2S RS=0 SR=0x00000001 | S2MM RS=1 SR=0x00000000
dma_bias: MM2S RS=1 SR=0x00000000 | S2MM RS=0 SR=0x00000000
dma_weight: send.running=True recv.running=True
dma_ifmap: send.running=True recv.running=True
dma_bias: send.running=True

=== VERIFIKASI ===
dma_weight: MM2S RS=1 | S2MM RS=1 | send.running=True
         recv.running=True
dma_ifmap: MM2S RS=1 | S2MM RS=1 | send.running=True
         recv.running=True
dma_bias: MM2S RS=1 | S2MM RS=0 | send.running=True


In [695]:
# ENSURES THE BIAS LEFT THE DMA
# 1. Send Async
h_bias = dma_send_async(dma_bias, pkt_b_l0)

# 2. Software-side Verification (Did the CPU write to DRAM correctly?)
# We use .flush() to ensure the data is pushed from CPU Cache to DRAM for the DMA to see
h_bias.flush() 
if np.array_equal(h_bias, pkt_b_l0):
    print("✅ [SW] Buffer Integrity: Verified (CPU -> DRAM)")

# 3. Hardware-side Verification (Wait for completion)
# We wait up to 2 seconds for the DMA to report IDLE
timeout = 2.0
start_time = time.time()
success = False

while (time.time() - start_time) < timeout:
    sr_val = dma_bias.read(0x04) # MM2S_SR
    is_idle = (sr_val >> 1) & 1
    is_halted = sr_val & 1
    
    if is_halted:
        print("❌ [HW] DMA ERROR: Send channel has HALTED!")
        break
    if is_idle:
        success = True
        break
    time.sleep(0.1)

# 4. Final Report
if success:
    # On many Xilinx DMAs, reading 0x28 after completion returns 0 (bytes remaining)
    # or the last length programmed.
    programmed_len = dma_bias.read(0x28)
    print(f"✅ [HW] Transfer Complete: DMA is IDLE.")
    print(f"   - Status Register: 0x{sr_val:08X}")
    print(f"   - Bytes Programmed: {programmed_len}")
else:
    print("⚠️ [HW] TIMEOUT: DMA is still busy. Your IP might not be asserting TREADY.")

# Keep the buffer alive until we are 100% sure we don't need to re-check
# h_bias.freebuffer()

✅ [SW] Buffer Integrity: Verified (CPU -> DRAM)
✅ [HW] Transfer Complete: DMA is IDLE.
   - Status Register: 0x00001002
   - Bytes Programmed: 32792


In [696]:
# ENSURES THE IFMAP LEFT THE DMA
# 1. Send Async (Trigger the transfer)
h_ifmap = dma_send_async(dma_ifmap, pkt_i_l0)

# 2. Software-side Verification: DRAM Consistency
h_ifmap.flush() 
if np.array_equal(h_ifmap, pkt_i_l0):
    print("✅ [SW] IFMAP Buffer Integrity: Verified (CPU -> DRAM)")
else:
    print("❌ [SW] IFMAP Buffer Mismatch: Data was corrupted during copy!")

# 3. Hardware-side Verification: AXI-Stream Handshake
# IFMAPs are larger, so we keep the 2-5 second timeout
timeout_ifmap = 3.0
start_time_i = time.time()
ifmap_success = False

while (time.time() - start_time_i) < timeout_ifmap:
    # Reading MM2S_SR (Status Register) for the IFMAP DMA
    sr_val_i = dma_ifmap.read(0x04) 
    is_idle_i = (sr_val_i >> 1) & 1
    is_halted_i = sr_val_i & 1
    
    if is_halted_i:
        # Check if bit 4 (DMAIntErr) or bit 5 (DMASlaveErr) are set
        print(f"❌ [HW] IFMAP DMA ERROR: Halted with SR 0x{sr_val_i:08X}")
        break
    if is_idle_i:
        ifmap_success = True
        break
    time.sleep(0.1)

# 4. Final Report
if ifmap_success:
    programmed_len_i = dma_ifmap.read(0x28) # MM2S_LENGTH
    print(f"✅ [HW] IFMAP Transfer Complete: DMA is IDLE.")
    print(f"   - Status Register: 0x{sr_val_i:08X}")
    print(f"   - Bytes Programmed: {programmed_len_i}")
else:
    # If it times out here, your IP is likely stuck waiting for something else 
    # (like weights or a start signal) before it accepts the IFMAP.
    print("⚠️ [HW] IFMAP TIMEOUT: DMA is still busy.")
    print(f"   - Current Status: 0x{sr_val_i:08X}")

# h_ifmap.freebuffer() # Only free after ensuring the layer execution is finished

✅ [SW] IFMAP Buffer Integrity: Verified (CPU -> DRAM)
✅ [HW] IFMAP Transfer Complete: DMA is IDLE.
   - Status Register: 0x00001002
   - Bytes Programmed: 32792


In [697]:
# print("===== LAYER 0 START =====")
# t0 = time.time()

# HDR   = 6
# WORDS = 4096

# pynq_proper_reset(dma_weight, "dma_weight", has_recv=True)
# pynq_proper_reset(dma_ifmap,  "dma_ifmap",  has_recv=True)
# pynq_proper_reset(dma_bias,   "dma_bias",   has_recv=False)

# print("Sending bias...")
# dma_send(dma_bias, pkt_b_l0)
# print("Sending ifmap...")
# dma_send(dma_ifmap, pkt_i_l0)
# print("Static data OK.")

# # --- Kirim semua weight batch (blocking per batch) ---
# print("Sending 8 weight batches...")
# for b in range(8):
#     print(f"  Batch {b}...", end=" ", flush=True)
#     dma_send(dma_weight, pkt_w_l0[b])
#     print("OK")

# print("All weights sent. FPGA sedang compute...")

# # --- Arm output buffers SETELAH semua weight terkirim ---
# buf_m0 = allocate(shape=(HDR + WORDS,), dtype=np.uint32)
# buf_m1 = allocate(shape=(WORDS,),       dtype=np.uint32)
# buf_m0[:] = 0
# buf_m1[:] = 0

# print("Arming output channels...")
# dma_weight.recvchannel.transfer(buf_m0)   # M0 = weight wrapper output
# dma_ifmap.recvchannel.transfer(buf_m1)    # M1 = ifmap wrapper output

# # --- Monitor sambil tunggu ---
# print("Waiting for results (max 30s)...")
# start = time.time()
# done_m0 = False
# done_m1 = False

# while time.time() - start < 30:
#     sr0 = dma_weight.read(0x34)
#     sr1 = dma_ifmap.read(0x34)
#     idle0 = (sr0 >> 1) & 1
#     idle1 = (sr1 >> 1) & 1
#     err0  = (sr0 >> 4) & 7  # bits [6:4]
#     err1  = (sr1 >> 4) & 7
    
#     print(f"  T={time.time()-start:.1f}s | M0: SR=0x{sr0:08X} idle={idle0} err={err0} | M1: SR=0x{sr1:08X} idle={idle1} err={err1}")
    
#     if err0 or err1:
#         print("  ERROR FLAG DETECTED — stopping")
#         break
    
#     if idle0 and idle1:
#         done_m0 = True
#         done_m1 = True
#         print("  Both channels idle — data received!")
#         break
    
#     time.sleep(1)

# # --- Baca hasil ---
# if done_m0 and done_m1:
#     magic    = int(buf_m0[0]) & 0xFFFF
#     layer_id = int(buf_m0[2]) & 0x3
#     print(f"\nMagic=0x{magic:04X} ({'OK' if magic == 0xDA7A else 'MISMATCH'}), Layer={layer_id}")
    
#     def sx24(arr):
#         a = arr.astype(np.int32) & 0xFFFFFF
#         return np.where(a >= 0x800000, a - 0x1000000, a)
    
#     m0 = sx24(buf_m0[HDR:])
#     m1 = sx24(buf_m1)
#     print(f"m0 non-zeros: {np.count_nonzero(m0)}")
#     print(f"m1 non-zeros: {np.count_nonzero(m1)}")
    
#     out_l0 = decode_output(m0, m1, num_ch=128, num_pos=64)
# #     save_output_perchannel(out_l0, "layer0_output_perchannel.txt", "LAYER 0")
#     print("Saved.")
# else:
#     print("\nTIMEOUT atau ERROR — cek ILA")
#     sr0 = dma_weight.read(0x34)
#     sr1 = dma_ifmap.read(0x34)
#     print(f"Final M0 SR: 0x{sr0:08X}")
#     print(f"Final M1 SR: 0x{sr1:08X}")

# buf_m0.freebuffer()
# buf_m1.freebuffer()
# print(f"===== LAYER 0 DONE ({time.time()-t0:.2f}s) =====")

In [698]:
dma_weight.register_map

RegisterMap {
  MM2S_DMACR = Register(RS=1, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  MM2S_DMASR = Register(Halted=0, Idle=0, SGIncld=0, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SGDecErr=0, IOC_Irq=0, Dly_Irq=0, Err_Irq=0, IRQThresholdSts=0, IRQDelaySts=0),
  MM2S_CURDESC = Register(Current_Descriptor_Pointer=0),
  MM2S_CURDESC_MSB = Register(Current_Descriptor_Pointer=0),
  MM2S_TAILDESC = Register(Tail_Descriptor_Pointer=0),
  MM2S_TAILDESC_MSB = Register(Tail_Descriptor_Pointer=0),
  MM2S_SA = Register(Source_Address=0),
  MM2S_SA_MSB = Register(Source_Address=0),
  MM2S_LENGTH = Register(Length=0),
  SG_CTL = Register(SG_CACHE=0, SG_USER=0),
  S2MM_DMACR = Register(RS=1, Reset=0, Keyhole=0, Cyclic_BD_Enable=0, IOC_IrqEn=0, Dly_IrqEn=0, Err_IrqEn=0, IRQThreshold=1, IRQDelay=0),
  S2MM_DMASR = Register(Halted=0, Idle=0, SGIncld=0, DMAIntErr=0, DMASlvErr=0, DMADecErr=0, SGIntErr=0, SGSlvErr=0, SG

In [699]:
# ============================================================
# CELL: FULL DMA RECOVERY — Jalankan ini, lalu restart kernel jika perlu
# ============================================================

def hard_start_channel(dma, channel_name):
    """
    Paksa start channel via register langsung, bypass PYNQ state machine.
    channel_name: 'send' atau 'recv'
    """
    if channel_name == 'send':
        cr_offset = 0x00
        sr_offset = 0x04
    else:
        cr_offset = 0x30
        sr_offset = 0x34

    # 1. Reset channel
    dma.write(cr_offset, 0x4)
    timeout = 0
    while (dma.read(cr_offset) & 0x4) and timeout < 10000:
        timeout += 1
    
    # 2. Set RS=1 (Run/Stop bit) langsung ke register
    current_cr = dma.read(cr_offset)
    dma.write(cr_offset, current_cr | 0x1)
    
    sr = dma.read(sr_offset)
    running = (dma.read(cr_offset) & 0x1) == 1
    halted  = (sr & 0x1)
    print(f"  {channel_name}: CR=0x{dma.read(cr_offset):08X} SR=0x{sr:08X} running={running} halted={halted}")

def full_dma_reset(dma, name, has_recv=True):
    print(f"Resetting {name}...")
    hard_start_channel(dma, 'send')
    if has_recv:
        hard_start_channel(dma, 'recv')
    
    # 3. Sync PYNQ internal state — trick: akses _reset langsung
    try:
        dma.sendchannel._flush_before_transfer = True
        dma.sendchannel._first_transfer = True
        if has_recv and dma.recvchannel is not None:
            dma.recvchannel._flush_before_transfer = True
            dma.recvchannel._first_transfer = True
    except Exception as e:
        print(f"  PYNQ state sync warning: {e}")

# ---- Jalankan ----
full_dma_reset(dma_weight, "dma_weight", has_recv=True)
full_dma_reset(dma_ifmap,  "dma_ifmap",  has_recv=True)
full_dma_reset(dma_bias,   "dma_bias",   has_recv=False)

# ---- Verifikasi ----
print("\n=== VERIFIKASI AKHIR ===")
for name, dma in [("dma_weight", dma_weight), ("dma_ifmap", dma_ifmap), ("dma_bias", dma_bias)]:
    mm2s_cr = dma.read(0x00)
    s2mm_cr = dma.read(0x30)
    mm2s_sr = dma.read(0x04)
    s2mm_sr = dma.read(0x34)
    print(f"{name}: MM2S RS={mm2s_cr&1} SR=0x{mm2s_sr:08X} | S2MM RS={s2mm_cr&1} SR=0x{s2mm_sr:08X}")
    
    # Cek error flags
    if mm2s_sr & 0x70:
        print(f"  ⚠ MM2S ERROR FLAGS: IntErr={(mm2s_sr>>4)&1} SlvErr={(mm2s_sr>>5)&1} DecErr={(mm2s_sr>>6)&1}")
    if s2mm_sr & 0x70:
        print(f"  ⚠ S2MM ERROR FLAGS: IntErr={(s2mm_sr>>4)&1} SlvErr={(s2mm_sr>>5)&1} DecErr={(s2mm_sr>>6)&1}")

Resetting dma_weight...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0
  recv: CR=0x00010003 SR=0x00000000 running=True halted=0
Resetting dma_ifmap...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0
  recv: CR=0x00010003 SR=0x00000000 running=True halted=0
Resetting dma_bias...
  send: CR=0x00010003 SR=0x00000000 running=True halted=0

=== VERIFIKASI AKHIR ===
dma_weight: MM2S RS=0 SR=0x00000001 | S2MM RS=1 SR=0x00000000
dma_ifmap: MM2S RS=0 SR=0x00000001 | S2MM RS=1 SR=0x00000000
dma_bias: MM2S RS=1 SR=0x00000000 | S2MM RS=0 SR=0x00000000


In [700]:
# ── Targeted inspection ───────────────────────────────────────────────
import pprint

# 1. Show keys available in ip_dict entries
first_ip = next(iter(overlay.ip_dict.values()))
print("=== ip_dict entry keys ===")
print(list(first_ip.keys()))

# 2. Show base address of each IP (try common key names)
print("\n=== IP Addresses ===")
for name, ip in overlay.ip_dict.items():
    addr = ip.get('phys_addr') or ip.get('base_addr') or ip.get('addr') or ip.get('mem_range', {})
    print(f"  {name:40s}  addr={addr}")

# 3. Look for anything reset/gpio related in overlay attributes
print("\n=== Reset/GPIO related attributes ===")
for attr in dir(overlay):
    if any(kw in attr.lower() for kw in ['gpio', 'reset', 'rst', 'ctrl', 'control']):
        print(f"  {attr} = {getattr(overlay, attr)}")

=== ip_dict entry keys ===
['type', 'mem_id', 'memtype', 'gpio', 'interrupts', 'parameters', 'registers', 'driver', 'device', 'state', 'bdtype', 'phys_addr', 'addr_range', 'fullpath']

=== IP Addresses ===
  axi_dma_0                                 addr=1077936128
  axi_dma_1                                 addr=1078001664
  axi_dma_2                                 addr=1078067200
  processing_system7_0                      addr={}

=== Reset/GPIO related attributes ===
  gpio_dict = {}
  interrupt_controllers = {}
  reset = <bound method Overlay.reset of <pynq.overlay.Overlay object at 0xacbf4ee0>>


In [701]:
# ── Fixed reset_dma_safe: use register-level soft reset ──────────────
# PYNQ's DMA class doesn't expose .reset() directly.
# AXI DMA soft reset: write 0x4 to MM2S_DMACR (0x00) and S2MM_DMACR (0x30)
# The reset bit self-clears when done. Poll until clear (max 100 attempts).
def reset_dma_safe(dma, name, has_s2mm=True):
    # MM2S reset
    dma.write(0x00, 0x4)
    for _ in range(100):
        if not (dma.read(0x00) & 0x4): break
        time.sleep(0.001)
    else:
        print(f"  WARNING: {name} MM2S reset did not self-clear")

    # S2MM reset
    if has_s2mm:
        dma.write(0x30, 0x4)
        for _ in range(100):
            if not (dma.read(0x30) & 0x4): break
            time.sleep(0.001)
        else:
            print(f"  WARNING: {name} S2MM reset did not self-clear")

    # Start channels via PYNQ API
    dma.sendchannel.start()
    if has_s2mm:
        dma.recvchannel.start()

    # SR=0 is normal right after reset+start before any transfer.
    # Check MM2S_DMACR (0x00) instead — bit0=RS should be 1 if running.
    mm2s_cr = dma.read(0x00)
    s2mm_cr = dma.read(0x30) if has_s2mm else None
    mm2s_sr = dma.read(0x04)
    s2mm_sr = dma.read(0x34) if has_s2mm else None

    print(f"  {name}:")
    print(f"    MM2S CR=0x{mm2s_cr:08X} SR=0x{mm2s_sr:08X}  RS(running)={(mm2s_cr>>0)&1}")
    if has_s2mm:
        print(f"    S2MM CR=0x{s2mm_cr:08X} SR=0x{s2mm_sr:08X}  RS(running)={(s2mm_cr>>0)&1}")

    # OK = RS bit is set (DMA is running), not based on SR value
    ok = bool((mm2s_cr >> 0) & 1)
    if has_s2mm:
        ok = ok and bool((s2mm_cr >> 0) & 1)
    print(f"    → {'OK' if ok else 'FAILED — RS bit not set'}")
    return ok

In [702]:
# # ── CELL: Master reset — run this before EVERY layer execution ────────
# # overlay.reset() is the only guaranteed clean slate since there is
# # no aresetn GPIO exposed in this bitstream.
# print("Reloading bitstream...")
# overlay.reset()

# # Re-bind after reload
# dma_weight = overlay.axi_dma_0
# dma_ifmap  = overlay.axi_dma_1
# dma_bias   = overlay.axi_dma_2
# print("Overlay reloaded. Re-binding DMAs...")

# # Now soft-reset the DMAs on top of the fresh PL state
# ok0 = reset_dma_safe(dma_weight, "dma_weight", has_s2mm=True)
# ok1 = reset_dma_safe(dma_ifmap,  "dma_ifmap",  has_s2mm=True)
# ok2 = reset_dma_safe(dma_bias,   "dma_bias",   has_s2mm=False)
# if not (ok0 and ok1 and ok2):
#     raise RuntimeError("DMA failed to initialize — do not proceed")

# print("Full reset complete — hardware and DMAs are in a clean state.")
# print("Safe to run layer execution now.")

In [703]:
def full_reset(overlay_obj, dma_map):
    """
    Full PL reset: reloads bitstream + soft-resets and starts all DMAs.

    Parameters
    ----------
    overlay_obj : pynq.Overlay — the loaded overlay
    dma_map     : list of (attr_name, display_name, has_s2mm) tuples
                  e.g. [("axi_dma_0", "dma_weight", True), ...]

    Returns
    -------
    dict of {display_name: dma_object} — rebound DMA handles
    """
    print("Reloading bitstream...")
    overlay_obj.reset()
    print("Overlay reloaded. Re-binding DMAs...")

    dmas = {}
    for attr_name, display_name, has_s2mm in dma_map:
        dma = getattr(overlay_obj, attr_name)
        ok  = reset_dma_safe(dma, display_name, has_s2mm=has_s2mm)
        if not ok:
            raise RuntimeError(f"DMA '{display_name}' failed to initialize — do not proceed")
        dmas[display_name] = dma

    print("Full reset complete — hardware and DMAs are in a clean state.")
    return dmas


# ── DMA map: (overlay_attr, display_name, has_s2mm) ──────────────────
DMA_MAP = [
    ("axi_dma_0", "dma_weight", True),
    ("axi_dma_1", "dma_ifmap",  True),
    ("axi_dma_2", "dma_bias",   False),
]

# ── Usage ─────────────────────────────────────────────────────────────
dmas = full_reset(overlay, DMA_MAP)
dma_weight = dmas["dma_weight"]
dma_ifmap  = dmas["dma_ifmap"]
dma_bias   = dmas["dma_bias"]

Reloading bitstream...
Overlay reloaded. Re-binding DMAs...
  dma_weight:
    MM2S CR=0x00010003 SR=0x00000000  RS(running)=1
    S2MM CR=0x00010003 SR=0x00000000  RS(running)=1
    → OK
  dma_ifmap:
    MM2S CR=0x00010003 SR=0x00000000  RS(running)=1
    S2MM CR=0x00010003 SR=0x00000000  RS(running)=1
    → OK
  dma_bias:
    MM2S CR=0x00010003 SR=0x00000000  RS(running)=1
    → OK
Full reset complete — hardware and DMAs are in a clean state.


In [704]:
def run_layer_process(dma_bias, dma_ifmap, dma_weight,
                      pkt_bias, pkt_ifmap, pkt_weights,
                      num_batches=8, data_words=4096, header_words=6,
                      timeout=30):
    """
    Run a single layer inference on hardware.
    Returns (m0, m1) raw decoded int32 arrays, or (None, None) if failed.
    """
    total_words    = header_words + data_words
    results_buf_m0 = allocate(shape=(total_words,), dtype=np.uint32)
    results_buf_m1 = allocate(shape=(total_words,), dtype=np.uint32)
    results_buf_m0[:] = 0
    results_buf_m1[:] = 0

    try:
        # 1. Send static data
        print("Sending Bias and IFMAP...")
        h_bias  = dma_send_async(dma_bias,  pkt_bias)
        h_ifmap = dma_send_async(dma_ifmap, pkt_ifmap)
        dma_bias.sendchannel.wait();  h_bias.freebuffer()
        dma_ifmap.sendchannel.wait(); h_ifmap.freebuffer()

        # 2. Arm recv channels BEFORE sending weights
        print("Arming recv channels...")
        dma_weight.recvchannel.transfer(results_buf_m0)
        dma_ifmap.recvchannel.transfer(results_buf_m1)

        # 3. Send weight batches
        print(f"Sending {num_batches} weight batches...")
        for b in range(num_batches):
            print(f"  Batch {b}...", end=" ", flush=True)
            h_w = dma_send_async(dma_weight, pkt_weights[b])
            dma_weight.sendchannel.wait(); h_w.freebuffer()
            print(f"OK | MM2S_SR=0x{dma_weight.read(0x04):08X}")

        # 4. Wait for DA7A result
        print("Waiting for DA7A output...")
        start = time.time()
        done  = False
        while time.time() - start < timeout:
            sr0   = dma_weight.read(0x34); sr1   = dma_ifmap.read(0x34)
            idle0 = (sr0 >> 1) & 1;        idle1 = (sr1 >> 1) & 1
            err0  = (sr0 >> 4) & 7;        err1  = (sr1 >> 4) & 7
            len0  = dma_weight.read(0x58); len1  = dma_ifmap.read(0x58)
            print(f"  T={time.time()-start:.1f}s | "
                  f"M0: SR=0x{sr0:08X} idle={idle0} err={err0} len={len0}B | "
                  f"M1: SR=0x{sr1:08X} idle={idle1} err={err1} len={len1}B")
            if err0 or err1: print("  ERROR — stopping"); break
            if idle0 and idle1: done = True; print("  Done!"); break
            time.sleep(1)

        # 5. Debug status
        sr0 = dma_weight.read(0x34); sr1 = dma_ifmap.read(0x34)
        print(f"M0 S2MM SR: 0x{sr0:08X}  IntErr={(sr0>>4)&1}  SlvErr={(sr0>>5)&1}")
        print(f"M1 S2MM SR: 0x{sr1:08X}  IntErr={(sr1>>4)&1}  SlvErr={(sr1>>5)&1}")

        if not done:
            print("Transfer did not complete.")
            return None, None

        # 6. Validate header
        magic    = int(results_buf_m0[0]) & 0xFFFF
        layer_id = int(results_buf_m0[2]) & 0x3
        print(f"\nHeader: Magic=0x{magic:04X} ({'OK' if magic == 0xDA7A else 'MISMATCH'}), Layer={layer_id}")
        for i in range(header_words):
            print(f"  buf_m0[{i}] = 0x{int(results_buf_m0[i]):08X}")
        if magic != 0xDA7A:
            print("ERROR: Magic mismatch — aborting.")
            return None, None

        # 7. Sign-extend 24-bit values
        def sx24(arr):
            a = arr.astype(np.int32) & 0xFFFFFF
            return np.where(a >= 0x800000, a - 0x1000000, a)

        m0 = sx24(results_buf_m0[header_words:])
        m1 = sx24(results_buf_m1[header_words:])
        print(f"m0 non-zeros: {np.count_nonzero(m0)} | m1 non-zeros: {np.count_nonzero(m1)}")
        return m0, m1

    finally:
        results_buf_m0.freebuffer()
        results_buf_m1.freebuffer()

def get_layer_output(m0, m1, num_ch, num_pos, layer_name="LAYER", data_words=4096):
    """
    Decode (m0, m1) BRAM-interleaved arrays into a [num_ch, num_pos] array
    and print a corner summary.
    Returns hardware_out [num_ch, num_pos] int32.
    """
    bram_depth = data_words // 8   # 512 for 4096 / 8 streams

    def decode_val(ch, pos):
        bram_id   = ch % 16
        page      = ch // 16
        bram_addr = page * num_pos + pos
        if bram_id < 8:
            return int(m0[bram_id * bram_depth + bram_addr])
        else:
            return int(m1[(bram_id - 8) * bram_depth + bram_addr])

    hardware_out = np.array([[decode_val(ch, pos)
                               for pos in range(num_pos)]
                              for ch in range(num_ch)], dtype=np.int32)
    
    print(f"Output shape: {hardware_out.shape}")
    
    print("\n=============================================================")
    print(f"{layer_name} OUTPUT: 128 Channels x 64 Positions")
    print("=============================================================")
    for grp in range(num_ch // 8):                          # 16 groups × 8 ch = 128
        grp_start = grp * 8
        print(f"\n--- Channels {grp_start:3d} - {grp_start+7:3d} ---")
        # Header
        hdr = " Pos |" + "".join(f" Ch{grp_start+c:3d} |" for c in range(8))
        print(hdr)
        print("-----|" + "-------|" * 8)
        # Data rows
        for pos in range(NUM_POS):
            row = f" {pos:3d} |"
            for c in range(8):
                val = decode_val(grp_start + c, pos)
                row += f" {val:5d} |"
            print(row)
    print("\n=============================================================")
    return hardware_out

In [705]:
m0_l0, m1_l0 = run_layer_process(
    dma_bias    = dma_bias,
    dma_ifmap   = dma_ifmap,
    dma_weight  = dma_weight,
    pkt_bias    = pkt_b_l0,
    pkt_ifmap   = pkt_i_l0,
    pkt_weights = pkt_w_l0,
    num_batches = 8,
)

Sending Bias and IFMAP...
Arming recv channels...
Sending 8 weight batches...
  Batch 0... OK | MM2S_SR=0x00001002
  Batch 1... OK | MM2S_SR=0x00001002
  Batch 2... OK | MM2S_SR=0x00001002
  Batch 3... OK | MM2S_SR=0x00001002
  Batch 4... OK | MM2S_SR=0x00001002
  Batch 5... OK | MM2S_SR=0x00001002
  Batch 6... OK | MM2S_SR=0x00001002
  Batch 7... OK | MM2S_SR=0x00001002
Waiting for DA7A output...
  T=0.0s | M0: SR=0x00001002 idle=1 err=0 len=16408B | M1: SR=0x00001002 idle=1 err=0 len=16408B
  Done!
M0 S2MM SR: 0x00001002  IntErr=0  SlvErr=0
M1 S2MM SR: 0x00001002  IntErr=0  SlvErr=0

Header: Magic=0xDA7A (OK), Layer=0
  buf_m0[0] = 0x0000DA7A
  buf_m0[1] = 0x00000002
  buf_m0[2] = 0x00000000
  buf_m0[3] = 0x00000000
  buf_m0[4] = 0x00000000
  buf_m0[5] = 0x00001000
m0 non-zeros: 514 | m1 non-zeros: 536


In [706]:
hardware_out_l0 = get_layer_output(m0_l0, m1_l0, num_ch=128, num_pos=64, layer_name="LAYER 0")

Output shape: (128, 64)

LAYER 0 OUTPUT: 128 Channels x 64 Positions

--- Channels   0 -   7 ---
 Pos | Ch  0 | Ch  1 | Ch  2 | Ch  3 | Ch  4 | Ch  5 | Ch  6 | Ch  7 |
-----|-------|-------|-------|-------|-------|-------|-------|-------|
   0 |     0 |     0 |   497 |     0 |     0 |     0 |     0 |     0 |
   1 |     0 |     0 |     0 |     0 |     0 |     0 |  9295 |     0 |
   2 |     0 |     0 | 15124 |     0 |     0 |     0 |     0 |     0 |
   3 |     0 |     0 |     0 |     0 |  6714 |     0 | 15680 |     0 |
   4 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |
   5 |     0 |     0 |     0 |     0 |  6180 |     0 |     0 |     0 |
   6 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |
   7 |     0 |     0 |     0 | 14845 | 33789 |     0 |     0 |     0 |
   8 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |     0 |
   9 |     0 |     0 |     0 | 12978 |     0 |     0 |     0 |     0 |
  10 |     0 |     0 |     0 | 21706 |     0 |     

In [707]:

# # 1. Prepare buffers for Final Results only — no more per-batch receipt buffers.
# #    Output_Manager now sends a single DA7A packet after all 8 batches complete.
# #    M0: 6-word header + 4096 data words. M1: 4096 data words.
# # 6 + 4096
# results_buf_m0 = allocate(shape=(6 + 4096,), dtype=np.uint32)
# results_buf_m1 = allocate(shape=(6 + 4096,),      dtype=np.uint32)
# results_buf_m0[:] = 0
# results_buf_m1[:] = 0

# print("===== STARTING LAYER 0 EXECUTION =====")

# # 2. SEND STATIC DATA (Bias and IFMAP)
# print("Sending Bias and IFMAP...")
# h_bias  = dma_send_async(dma_bias,  pkt_b_l0)
# h_ifmap = dma_send_async(dma_ifmap, pkt_i_l0)
# dma_bias.sendchannel.wait()
# dma_ifmap.sendchannel.wait()
# h_bias.freebuffer()
# h_ifmap.freebuffer()

# # 3. ARM RECV CHANNELS *BEFORE* SENDING WEIGHTS.
# #    There is now only one TLAST (at the end of all 8 batches).
# #    If we arm after sending, we risk missing the single transfer event.
# print("Arming recv channels for final DA7A output...")
# dma_weight.recvchannel.transfer(results_buf_m0)
# dma_ifmap.recvchannel.transfer(results_buf_m1)

# # 4. WEIGHT BATCH LOOP — fire and forget, no per-batch receipt expected
# for b in range(8):
#     print(f"--- Sending Batch {b} ---")
#     h_w = dma_send_async(dma_weight, pkt_w_l0[b])
#     dma_weight.sendchannel.wait()
#     h_w.freebuffer()
#     print(f"  Batch {b} sent | MM2S_SR=0x{dma_weight.read(0x04):08X}")

# # 5. WAIT FOR FINAL DA7A RESULT
# print("All batches sent. Waiting for final DA7A output...")
# OFFSET_SR  = 0x34
# OFFSET_LEN = 0x58
# start = time.time()
# done_m0 = done_m1 = False
# while time.time() - start < 30:
#     sr0   = dma_weight.read(OFFSET_SR)
#     sr1   = dma_ifmap.read(OFFSET_SR)
#     idle0 = (sr0 >> 1) & 1
#     idle1 = (sr1 >> 1) & 1
#     err0  = (sr0 >> 4) & 7
#     err1  = (sr1 >> 4) & 7
#     len0  = dma_weight.read(OFFSET_LEN)
#     len1  = dma_ifmap.read(OFFSET_LEN)
#     print(f"  T={time.time()-start:.1f}s | M0: SR=0x{sr0:08X} idle={idle0} err={err0} len={len0}B "
#           f"| M1: SR=0x{sr1:08X} idle={idle1} err={err1} len={len1}B")
#     if err0 or err1:
#         print("  ERROR — stopping")
#         break
#     if idle0 and idle1:
#         done_m0 = done_m1 = True
#         print("  Done!")
#         break
#     time.sleep(1)

# # 6. DEBUG STATUS
# print("\n--- DMA ERROR CHECK ---")
# sr0 = dma_weight.read(0x34)
# sr1 = dma_ifmap.read(0x34)
# print(f"M0 S2MM SR: 0x{sr0:08X}")
# print(f"  DMAIntErr  (TLAST mismatch): {(sr0>>4)&1}")
# print(f"  DMASlvErr  (Bus error):      {(sr0>>5)&1}")
# print(f"M1 S2MM SR: 0x{sr1:08X}")
# print(f"  DMAIntErr  (TLAST mismatch): {(sr1>>4)&1}")
# print(f"  DMASlvErr  (Bus error):      {(sr1>>5)&1}")
# print("-----------------------\n")

# # 7. PROCESS RESULTS
# if done_m0 and done_m1:
#     print("Header M0:")
#     for i in range(6):
#         print(f"  results_buf_m0[{i}] = 0x{int(results_buf_m0[i]):08X}")
#     magic    = int(results_buf_m0[0]) & 0xFFFF
#     layer_id = int(results_buf_m0[2]) & 0x3
#     print(f"Magic=0x{magic:04X} ({'OK' if magic == 0xDA7A else 'MISMATCH'}), Layer={layer_id}")

#     def sx24(arr):
#         a = arr.astype(np.int32) & 0xFFFFFF
#         return np.where(a >= 0x800000, a - 0x1000000, a)

#     m0 = sx24(results_buf_m0[6:])   # 4096 words
#     m1 = sx24(results_buf_m1[6:])        # 4096 words
#     print(f"m0 non-zeros: {np.count_nonzero(m0)}")
#     print(f"m1 non-zeros: {np.count_nonzero(m1)}")
#     data_section_1 = m0  # already sliced above, kept for compatibility
#     out_l0 = decode_output(m0, m1, num_ch=128, num_pos=64)
#     print("Decode done.")
# else:
#     print("Transfer did not complete — check errors above.")

# results_buf_m0.freebuffer()
# results_buf_m1.freebuffer()

In [708]:
# # ── Print in testbench format ─────────────────────────────────────
# NUM_CH  = 128
# NUM_POS = 64

# # Decode raw buffers exactly as the testbench does:
# #   BRAM_ID   = ch % 16
# #   page      = ch / 16
# #   BRAM_ADDR = page * 64 + pos
# #   val = m0_buf[BRAM_ID*512 + BRAM_ADDR]       if BRAM_ID < 8
# #         m1_buf[(BRAM_ID-8)*512 + BRAM_ADDR]   otherwise
# def decode_val(ch, pos, m0, m1):
#     bram_id   = ch % 16
#     page      = ch // 16
#     bram_addr = page * 64 + pos
#     if bram_id < 8:
#         return int(m0[bram_id * 512 + bram_addr])
#     else:
#         return int(m1[(bram_id - 8) * 512 + bram_addr])
    
# # Build [128 x 64] array
# hardware_out_l0 = np.array([[decode_val(ch, pos, m0, m1) 
#                      for pos in range(NUM_POS)] 
#                     for ch in range(NUM_CH)], dtype=np.int32)
    
# # ── Console print: groups of 8 channels (matches print_layer0_output_to_console) ──
# print("\n=============================================================")
# print("LAYER 0 OUTPUT: 128 Channels x 64 Positions")
# print("=============================================================")
# for grp in range(16):                          # 16 groups × 8 ch = 128
#     grp_start = grp * 8
#     print(f"\n--- Channels {grp_start:3d} - {grp_start+7:3d} ---")
#     # Header
#     hdr = " Pos |" + "".join(f" Ch{grp_start+c:3d} |" for c in range(8))
#     print(hdr)
#     print("-----|" + "-------|" * 8)
#     # Data rows
#     for pos in range(NUM_POS):
#         row = f" {pos:3d} |"
#         for c in range(8):
#             val = decode_val(grp_start + c, pos, m0, m1)
#             row += f" {val:5d} |"
#         print(row)
# print("\n=============================================================")

In [709]:
print(result0_2d[:10, :10])

[[    0     0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0     0 18292     0 12436     0  3387     0     0     0]
 [    0     0     0     0     0     0     0 11878     0  9742]
 [    0     0     0  6522     0  2216     0 32635     0     0]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0  8880     0  3507     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0]]


# Comparison

In [710]:
def rrmse(x_hat, x):
    x_hat = x_hat.astype(np.float64)
    x     = x.astype(np.float64)
    return np.sqrt(np.mean((x_hat - x) ** 2)) / (np.sqrt(np.mean(x ** 2)) + 1e-8)

def corrcoef(x_hat, x):
    x_hat = x_hat.astype(np.float64) - x_hat.mean()
    x     = x.astype(np.float64)     - x.mean()
    return np.sum(x_hat * x) / (
        np.sqrt(np.sum(x_hat ** 2)) * np.sqrt(np.sum(x ** 2)) + 1e-8
    )

def compare_hardware_vs_reference(hardware, reference, layer_name="LAYER 0",
                                   num_ch=128, rrmse_thresh=0.2, corr_thresh=0.90):
    assert hardware.shape == reference.shape, \
        f"Shape mismatch: hardware={hardware.shape}, reference={reference.shape}"

    q_rrmse  = rrmse(hardware, reference)
    q_corr   = corrcoef(hardware.flatten(), reference.flatten())
    ch_rrmse = [rrmse(hardware[ch], reference[ch]) for ch in range(num_ch)]
    ch_corr  = [corrcoef(hardware[ch], reference[ch]) for ch in range(num_ch)]

    worst_rrmse = sorted(range(num_ch), key=lambda c: ch_rrmse[c], reverse=True)[:5]
    worst_corr  = sorted(range(num_ch), key=lambda c: ch_corr[c])[:5]

    print("=============================================================")
    print(f"{layer_name} — Hardware vs Reference Comparison")
    print("=============================================================")
    print(f"  Shape : {hardware.shape}")
    print(f"  RRMSE : {q_rrmse:.6e}  ({'PASS ✓' if q_rrmse < rrmse_thresh else 'WARN ✗'})")
    print(f"  Corr  : {q_corr:.6f}   ({'PASS ✓' if q_corr  > corr_thresh  else 'WARN ✗'})")
    print(f"\n  --- Worst 5 by RRMSE ---")
    print(f"  {'ch':>4}  {'rrmse':>12}  {'corr':>10}")
    for ch in worst_rrmse:
        print(f"  {ch:4d}  {ch_rrmse[ch]:12.6e}  {ch_corr[ch]:10.6f}")
    print(f"\n  --- Worst 5 by Corr ---")
    print(f"  {'ch':>4}  {'rrmse':>12}  {'corr':>10}")
    for ch in worst_corr:
        print(f"  {ch:4d}  {ch_rrmse[ch]:12.6e}  {ch_corr[ch]:10.6f}")
    print("=============================================================")

    return {"rrmse": q_rrmse, "corr": q_corr, "ch_rrmse": ch_rrmse, "ch_corr": ch_corr,
            "pass": q_rrmse < rrmse_thresh and q_corr > corr_thresh}




In [711]:
# ── Usage ─────────────────────────────────────────────────────────────
metrics_l0 = compare_hardware_vs_reference(hardware_out_l0, result0_2d,
                                           layer_name="LAYER 0", num_ch=128
                                          )

LAYER 0 — Hardware vs Reference Comparison
  Shape : (128, 64)
  RRMSE : 1.984562e-01  (PASS ✓)
  Corr  : 0.978603   (PASS ✓)

  --- Worst 5 by RRMSE ---
    ch         rrmse        corr
   113  1.567768e+11    0.000000
    11  5.353750e+10    0.000000
    30  5.287500e+09    0.000000
   100  4.100000e+09    0.000000
    49  5.126471e+01    1.000000

  --- Worst 5 by Corr ---
    ch         rrmse        corr
    78  1.335932e+00   -0.015873
     8  0.000000e+00    0.000000
    11  5.353750e+10    0.000000
    12  0.000000e+00    0.000000
    14  1.000000e+00    0.000000
